In [3]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

class Country(BaseModel):
    """Information about a country"""
    name: str = Field(..., description="The name of the country")
    capital: str = Field(..., description="The capital city of the country")
    population: int = Field(..., description="The population of the country")
    area: float = Field(..., description="The total area of the country in square kilometers")


structured_llm = llm.with_structured_output(Country)

structured_llm



RunnableBinding(bound=ChatGoogleGenerativeAI(model='models/gemini-2.5-pro', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001E903ADBB10>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Country', 'description': 'Information about a country', 'parameters': {'properties': {'name': {'description': 'The name of the country', 'type': 'string'}, 'capital': {'description': 'The capital city of the country', 'type': 'string'}, 'population': {'description': 'The population of the country', 'type': 'integer'}, 'area': {'description': 'The total area of the country in square kilometers', 'type': 'number'}}, 'required': ['name', 'capital', 'population', 'area'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Country', 'description': 'In

In [8]:
structured_llm.invoke("Tell me about Canada.")

Country(name='Canada', capital='Ottawa', population=38010000, area=9985000.0)

In [17]:
from typing_extensions import Annotated, TypedDict
from typing import Optional

# Type Dict
class Joke(TypedDict):
    """Joke to tell user"""
    
    setup: Annotated[str, ..., 'The setup of the joke']
    
    #alternatively, we could have specified setup as:
    
    #setup: str                  # no default, no description
    #setup: Annotated[str, ...]  # no default, no description
    #setup: Annotated[str,"foo"] # default, no description
    
    punchline: Annotated[str, ..., 'The punchline of the joke']
    rating: Annotated[Optional[int], None, 'Rating of the joke from 1 to 10']

structured_llm = llm.with_structured_output(Joke)

structured_llm.invoke('Tell me a joke about dolphins.')
 

{'punchline': 'Nothing, he just waved.',
 'rating': 7.0,
 'setup': 'What did the dolphin say to the other dolphin?'}

In [19]:
json_schema = {
    "title": "joke",
    "description": "Joke to tell user.",
    "type": "object",
    "properties": {
        "setup": {
            "type": "string",
            "description": "The setup of the joke",
        },
        "punchline": {
            "type": "string",
            "description": "The punchline to the joke",
        },
        "rating": {
            "type": "integer",
            "description": "How funny the joke is, from 1 to 10",
            "default": None,
        },
    },
    "required": ["setup", "punchline"],
}
structured_llm = llm.with_structured_output(json_schema)

structured_llm.invoke("Tell me a joke about cats")

Key 'parameters' is not supported in schema, ignoring


{'punchline': 'Because she wanted to be a first-aid kit!',
 'rating': 7.0,
 'setup': 'Why did the cat join the Red Cross?'}